# Tracking LLM spend with LiteLLM

This notebook is a short, hands-on tutorial on how to monitor the **cost** of your LLM calls.

Contents:

1. Send a chat completion and read the **cost of that single request**.
2. Look up the **pricing** of the models available on the proxy.
3. Check the **total spend and budget** of your API key.
4. Get a **day-by-day breakdown** of your usage.

### Prerequisites

- Access to ERNI LiteLLM proxy instance.
- Two environment variables set before starting Jupyter:
  - `LITELLM_BASE_URL` – the URL of the proxy (e.g. `https://litellm.example.com`)
  - `LITELLM_API_KEY` – your virtual key (starts with `sk-...`)

## 1. Setup

Install `requests`, the only dependency we need to talk to the proxy's REST API.

In [ ]:
%pip install requests

Read the proxy URL and API key from the environment, and prepare the authentication header that every request will use.

In [ ]:
import os
import requests

LITELLM_API_KEY = os.environ.get("LITELLM_API_KEY")
LITELLM_BASE_URL = os.environ.get("LITELLM_BASE_URL")

if not LITELLM_API_KEY or not LITELLM_BASE_URL:
    raise ValueError("Please set the LITELLM_API_KEY and LITELLM_BASE_URL environment variables")

HEADERS = {"Authorization": f"Bearer {LITELLM_API_KEY}"}

## 2. Cost of a single request

LiteLLM exposes an **OpenAI-compatible** `/chat/completions` endpoint, so the request body is the same you would send to OpenAI.

On top of the normal response, the proxy adds the `x-litellm-response-cost` header: the cost (in USD) of that specific call, computed from the tokens used and the model's price.

In [ ]:
response = requests.post(
    f"{LITELLM_BASE_URL}/chat/completions",
    headers=HEADERS,
    json={
        "model": "gpt-5.6-terra",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Hey, how are you?"},
        ],
    },
)
data = response.json()

print("Answer:", data["choices"][0]["message"]["content"])
print("Tokens used:", data["usage"])
print("Cost (USD):", response.headers.get("x-litellm-response-cost"))

## 3. Model pricing

The `/model/info` endpoint lists the models you can use on the proxy, together with their metadata.
Look for fields such as `input_cost_per_token` and `output_cost_per_token` inside `model_info` to see how each model is priced.

In [ ]:
response = requests.get(f"{LITELLM_BASE_URL}/model/info", headers=HEADERS)
print(response.json())

## 4. Total spend of your key

The `/key/info` endpoint returns information about the key used in the request, including:

- `spend` – how much (USD) has been spent with this key so far
- `max_budget` – the spending limit, if one is set (`None` means unlimited)
- `models` – the models this key is allowed to call

In [ ]:
response = requests.get(f"{LITELLM_BASE_URL}/key/info", headers=HEADERS)
print(response.json())

## 5. Daily activity

The `/user/daily/activity` endpoint aggregates usage **per day** in a date range, with spend, tokens and number of requests, broken down by model, provider and API key.

Change `start_date` and `end_date` (format `YYYY-MM-DD`) to the period you are interested in.

In [ ]:

params = {
    "start_date": "2026-09-20",
    "end_date": "2026-10-22",
}

response = requests.get(
    f"{LITELLM_BASE_URL}/user/daily/activity",
    params=params,
    headers=HEADERS,
)
print(response.json())

## Recap

| What you want to know | Where to find it |
|---|---|
| Cost of one request | `x-litellm-response-cost` response header |
| Price of each model | `GET /model/info` |
| Total spend / budget of your key | `GET /key/info` |
| Usage day by day  splitted by model| `GET /user/daily/activity` |
